# Phase 1 — Legal Language Model Training

**Run this on a Kaggle GPU session** (T4, 16 GB VRAM).

Prerequisites:
- The tokenizer notebook has been committed and its output added as a
  Dataset (so `a1_tokenizer.py` and `bpe_out/bpe_40k.json` are accessible).

This notebook:
1. Loads the pre-trained BPE tokenizer
2. Streams, cleans, and tokenizes the corpus (temperature-scaled mixture, α=0.5)
3. Builds train/val datasets
4. Defines the GQA Transformer (RoPE, RMSNorm, SwiGLU, KV cache, weight tying)
5. Trains with mixed-precision + gradient accumulation
6. Saves checkpoints and the best model
7. Generates Phase 1 test predictions
8. Creates `submission.zip` for Codabench

In [23]:
import shutil

shutil.copy(
    "/kaggle/input/datasets/yuganjain/bpedata/bpe_40k.json",
    "/kaggle/working/bpe_40k.json"
)

shutil.copy(
    "/kaggle/input/datasets/yuganjain/bpedata/a1_tokenizer (2).py",
    "/kaggle/working/a1_tokenizer.py"
)

print("Files copied successfully.")

Files copied successfully.


In [24]:
# Add this in a cell before the training loop if you mounted it as a dataset
import shutil, os
src = "/kaggle/input/datasets/yuganjain/bpedata3/step_19000.pt"  # adjust name
dst = "/kaggle/working/checkpoints/step_19000.pt"
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)

'/kaggle/working/checkpoints/step_19000.pt'

In [1]:
print('hello')

hello


In [3]:
!pip -q install ijson tqdm huggingface_hub
import os, sys, re, json, html, unicodedata, time, math, random, pickle, zipfile
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import ijson
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.0 MB/s eta 0:00:00a 0:00:01
Imports OK
PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1 — Configuration (all hyperparameters in one place)

In [4]:
@dataclass
class Config:
    # ── Model ──
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19     # 150M params (with weight tying)
    n_query_heads: int   = 12
    n_kv_heads: int      = 4       # GQA ratio 3:1
    d_ff: int            = 2048    # SwiGLU intermediate
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10000.0

    # ── Training ──
    lr: float            = 3e-4
    min_lr: float        = 3e-5
    warmup_steps: int    = 500
    max_steps: int       = 30_000
    batch_size: int      = 8
    grad_accum: int      = 4       # effective batch = 32
    weight_decay: float  = 0.1
    max_grad_norm: float = 1.0
    betas: tuple         = (0.9, 0.95)
    use_amp: bool        = True

    # ── Data ──
    shares: dict         = None
    val_frac: float      = 0.02

    # ── Logging / checkpointing ──
    log_interval: int    = 50
    eval_interval: int   = 500
    save_interval: int   = 1000
    patience: int        = 5       # early stopping: stop after 5 evals without improvement
    seed: int            = 42

    # ── Generation ──
    gen_max_words: int        = 128
    gen_temperature: float    = 0.8
    gen_top_k: int            = 50
    gen_top_p: float          = 0.9
    gen_rep_penalty: float    = 1.1

    def __post_init__(self):
        if self.shares is None:
            # ── Temperature-scaled sampling (Conneau et al., 2020 — XLM-R) ──
            # Natural corpus proportions (measured from cleaned token counts):
            #   HC ≈ 55%, Trib ≈ 34%, SC ≈ 7%, Acts ≈ 4%
            # α=1.0 → natural, α=0.0 → equal, α=0.5 → square-root smoothing
            NATURAL = {"High Courts": 0.551, "Supreme Court": 0.073,
                       "Tribunals": 0.335, "Acts": 0.041}
            ALPHA = 0.5
            raw = {s: p ** ALPHA for s, p in NATURAL.items()}
            t = sum(raw.values())
            self.shares = {s: round(v / t, 2) for s, v in raw.items()}
            diff = round(1.0 - sum(self.shares.values()), 2)
            self.shares[max(self.shares, key=self.shares.get)] += diff
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads
        assert self.d_model % self.n_query_heads == 0
        assert self.n_query_heads % self.n_kv_heads == 0

    @property
    def eff_batch(self): return self.batch_size * self.grad_accum

CFG = Config()

# ── Paths ──
HF_REPO  = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {
    "Acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
OUT       = Path("/kaggle/working")
CKPT_DIR  = OUT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)

# Path to tokenizer from the tokenizer-notebook output (adjust if needed)
TOK_DATASET = "/kaggle/input"  # default dataset mount
TOK_CANDIDATES = [
    OUT / "bpe_out" / "bpe_40k.json",
    Path(TOK_DATASET) / "bpe-tokenizer-output" / "bpe_out" / "bpe_40k.json",
    Path(TOK_DATASET) / "bpe_out" / "bpe_40k.json",
    Path(TOK_DATASET) / "bpe_40k.json",
    Path("bpe_40k.json"),
]

print(f"Config: ~{CFG.d_model}d × {CFG.n_layers}L × {CFG.n_query_heads}Qh/{CFG.n_kv_heads}KVh, "
      f"eff_batch={CFG.eff_batch}, max_steps={CFG.max_steps}")
print(f"Mixture (α=0.5): {CFG.shares}")

Config: ~768d × 19L × 12Qh/4KVh, eff_batch=32, max_steps=20000
Mixture (α=0.5): {'High Courts': 0.42, 'Supreme Court': 0.15, 'Tribunals': 0.32, 'Acts': 0.11}


## 2 — Load tokenizer

In [5]:
# Find the tokenizer module
for candidate in [OUT, Path(TOK_DATASET), Path(".")]:
    mod_path = candidate / "a1_tokenizer.py"
    if mod_path.exists():
        sys.path.insert(0, str(candidate))
        break
else:
    # Write it inline as fallback (same code as the tokenizer notebook)
    raise FileNotFoundError(
        "a1_tokenizer.py not found. Add the tokenizer notebook output as a Dataset, "
        "or place a1_tokenizer.py in /kaggle/working/."
    )

from a1_tokenizer import BPETokenizer

# Find the trained tokenizer JSON
tok_path = None
for c in TOK_CANDIDATES:
    if c.exists():
        tok_path = c; break
assert tok_path is not None, (
    "Trained tokenizer JSON not found. Checked:\n" +
    "\n".join(f"  {c}" for c in TOK_CANDIDATES)
)

tokenizer = BPETokenizer.load(str(tok_path))
CFG.vocab_size = tokenizer.vocab_size
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

# Quick smoke test
_test = "The Supreme Court held that the appellant was entitled to relief."
assert tokenizer.decode(tokenizer.encode(_test)) == _test, "Round-trip failed"
print("Tokenizer smoke test passed")

Tokenizer loaded: vocab_size=40000
Tokenizer smoke test passed


## 3 — Data download + cleaning utilities

In [6]:
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
LOCAL = {}
for src, fn in HF_FILES.items():
    LOCAL[src] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"{src:14s} {Path(LOCAL[src]).stat().st_size/2**30:.2f} GB")
test_path = hf_hub_download(HF_REPO, TEST_FILE, repo_type="dataset", token=HF_TOKEN)
print(f"{'Test':14s} -> {test_path}")

# ── Cleaning (same as tokenizer notebook) ──
TAG_RE   = re.compile(r"</?[A-Za-z][^<>]{0,200}>")
URL_RE   = re.compile(r"https?://\S+|www\.\S+", re.I)
ENT_RE   = re.compile(r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);", re.I)
CTRL_RE  = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WSP_RE   = re.compile(r"[\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000]")
ZW_RE    = re.compile(r"[\u200b\u200c\u200d\u2060]")
SPACE_RE = re.compile(r"\s+")
MAX_REPL = 50
MIN_CHARS = 50

def clean_text(text):
    if text is None: return ""
    if isinstance(text, list): text = " ".join(map(str, text))
    elif isinstance(text, dict): text = " ".join(map(str, text.values()))
    else: text = str(text)
    if text.count("\ufffd") > MAX_REPL: return ""
    text = TAG_RE.sub(" ", text)
    text = html.unescape(text)
    text = URL_RE.sub(" ", text)
    text = ENT_RE.sub(" ", text)
    text = text.replace("\ufeff", " ").replace("\ufffd", " ")
    text = CTRL_RE.sub(" ", text)
    text = WSP_RE.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    text = ZW_RE.sub("", text)
    return SPACE_RE.sub(" ", text).strip()

TEXT_KEYS = ("text", "content", "judgment", "document", "body")
def extract_text(rec):
    if isinstance(rec, str): return rec
    if isinstance(rec, dict):
        for k in TEXT_KEYS:
            v = rec.get(k)
            if isinstance(v, str): return v
            if isinstance(v, list): return " ".join(map(str, v))
        return None
    return None

def iter_records(path):
    with open(path, "rb") as f:
        first = f.read(4096).lstrip()[:1]
    with open(path, "rb") as f:
        if first == b"[": yield from ijson.items(f, "item")
        elif first == b"{":
            for _, v in ijson.kvitems(f, ""): yield v
        else: raise ValueError(path)

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Acts           0.68 GB


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

High Courts    9.94 GB


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Supreme Court  1.23 GB


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Tribunals      6.04 GB


test.json: 0.00B [00:00, ?B/s]

Test           -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


## 4 — Estimate corpus budget (bottleneck-driven) → stream & tokenize

We estimate each source's cleaned size from file size, find the **bottleneck
source** (the one that runs out first under the temperature-scaled shares),
and derive `TARGET_CHARS` from it. This naturally yields ~2.1 B tokens —
Chinchilla-optimal for ~106 M parameters.

In [7]:
# ── 4a: Bottleneck-driven budget ──
CLEAN_RATIO = 0.85
sizes = {s: Path(p).stat().st_size for s, p in LOCAL.items()}
est_chars = {s: int(sizes[s] * CLEAN_RATIO) for s in sizes}

budgets_if = {src: est_chars[src] / CFG.shares[src] for src in CFG.shares}
bottleneck = min(budgets_if, key=budgets_if.get)
TARGET_CHARS = int(budgets_if[bottleneck])

print("Estimated cleaned chars per source:")
for src in sizes:
    print(f"  {src:14s} {est_chars[src]/1e9:.2f}B  (share={CFG.shares[src]:.0%}  "
          f"budget_if_bottleneck={budgets_if[src]/1e9:.2f}B)")
print(f"\nBottleneck source: {bottleneck}")
print(f"TARGET_CHARS:      {TARGET_CHARS:,} ({TARGET_CHARS/1e9:.2f}B)")

# ── 4b: Stream, clean, tokenize, split ──
rng = random.Random(CFG.seed)
train_ids_all = []
val_ids_all   = []
stats = {}

print("\nStreaming + tokenizing...")
for src, path in LOCAL.items():
    budget = TARGET_CHARS * CFG.shares[src]
    p = min(1.0, budget / (CLEAN_RATIO * sizes[src]))
    chars = docs = train_toks = val_toks = 0

    for rec in tqdm(iter_records(path), desc=src):
        if p < 1.0 and rng.random() > p: continue
        raw = extract_text(rec)
        if raw is None: continue
        text = clean_text(raw)
        if len(text) < MIN_CHARS: continue

        ids = tokenizer.encode(text, add_special_tokens=True)

        if rng.random() < CFG.val_frac:
            val_ids_all.extend(ids)
            val_toks += len(ids)
        else:
            train_ids_all.extend(ids)
            train_toks += len(ids)

        chars += len(text); docs += 1
        if chars >= budget * 1.05: break

    stats[src] = {"docs": docs, "chars": chars, "train_toks": train_toks, "val_toks": val_toks}
    print(f"  {src}: {docs:,} docs, {chars/1e6:.0f}M chars, "
          f"{train_toks:,} train toks, {val_toks:,} val toks "
          f"(p={p:.2%})")

print(f"\nTotal train tokens: {len(train_ids_all):,}")
print(f"Total val tokens:   {len(val_ids_all):,}")

# Realised mix
tot_c = sum(s["chars"] for s in stats.values())
print("\nRealised mix:")
for src, s in stats.items():
    print(f"  {src:14s} {s['chars']/tot_c:.1%}  (target {CFG.shares[src]:.0%})")

# Convert to numpy
train_arr = np.array(train_ids_all, dtype=np.int32)
val_arr   = np.array(val_ids_all,   dtype=np.int32)
del train_ids_all, val_ids_all
print(f"\nArrays: train={train_arr.nbytes/1e6:.0f} MB, val={val_arr.nbytes/1e6:.0f} MB")

Estimated cleaned chars per source:
  Acts           0.62B  (share=11%  budget_if_bottleneck=5.63B)
  High Courts    9.07B  (share=42%  budget_if_bottleneck=21.60B)
  Supreme Court  1.12B  (share=15%  budget_if_bottleneck=7.49B)
  Tribunals      5.51B  (share=32%  budget_if_bottleneck=17.22B)

Bottleneck source: Acts
TARGET_CHARS:      5,631,650,972 (5.63B)

Streaming + tokenizing...


Acts: 0it [00:00, ?it/s]

  Acts: 16,847 docs, 650M chars, 119,508,926 train toks, 2,397,901 val toks (p=100.00%)


High Courts: 0it [00:00, ?it/s]

  High Courts: 648,855 docs, 2484M chars, 455,447,345 train toks, 9,822,831 val toks (p=26.07%)


Supreme Court: 0it [00:00, ?it/s]

  Supreme Court: 40,048 docs, 887M chars, 163,544,870 train toks, 3,410,294 val toks (p=75.14%)


Tribunals: 0it [00:00, ?it/s]

  Tribunals: 269,410 docs, 1892M chars, 369,377,018 train toks, 7,861,779 val toks (p=32.70%)

Total train tokens: 1,107,878,159
Total val tokens:   23,492,805

Realised mix:
  Acts           11.0%  (target 11%)
  High Courts    42.0%  (target 42%)
  Supreme Court  15.0%  (target 15%)
  Tribunals      32.0%  (target 32%)

Arrays: train=4432 MB, val=94 MB


## 5 — Dataset & DataLoader

In [8]:
class BlockDataset(Dataset):
    """Fixed-length blocks for causal LM. x=[i:i+L], y=[i+1:i+L+1]."""
    def __init__(self, data: np.ndarray, block_len: int):
        self.data = data
        self.block_len = block_len
        self.n = max(0, (len(data) - 1) // block_len)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        i = idx * self.block_len
        x = torch.from_numpy(self.data[i   : i + self.block_len].astype(np.int64))
        y = torch.from_numpy(self.data[i+1 : i + self.block_len + 1].astype(np.int64))
        return x, y

train_ds = BlockDataset(train_arr, CFG.max_seq_len)
val_ds   = BlockDataset(val_arr,   CFG.max_seq_len)

train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True, drop_last=True)

print(f"Train: {len(train_ds):,} blocks ({len(train_ds)*CFG.max_seq_len:,} tokens)")
print(f"Val:   {len(val_ds):,} blocks")
print(f"Steps/epoch ≈ {len(train_ds) // CFG.batch_size // CFG.grad_accum:,}")

Train: 2,163,824 blocks (1,107,877,888 tokens)
Val:   45,884 blocks
Steps/epoch ≈ 67,619


## 6 — Model architecture

Decoder-only Transformer with:
- **Grouped Query Attention (GQA)**
- **Rotary Positional Embeddings (RoPE)**
- **RMSNorm** (pre-norm)
- **SwiGLU** feed-forward
- **KV cache** for generation
- **Weight tying** (embedding ↔ lm_head)

In [9]:
# ── RMSNorm ──
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

# ── RoPE ──
def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_full = c.expand_as(x1).repeat(1,1,1,2) if hd > 0 else c
    s_full = s.expand_as(x1).repeat(1,1,1,2) if hd > 0 else s
    # Proper expansion
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

# ── GQA ──
class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd)
            k = k.reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd)
            v = v.reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

# ── SwiGLU FFN ──
class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

# ── Transformer block ──
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

# ── Full model ──
class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        m = torch.full((T, S), float("-inf"), device=device)
        off = S - T
        for i in range(T):
            m[i, :off+i+1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, ids, max_new=256, temp=0.8, top_k=50, top_p=0.9,
                 rep_pen=1.1, stop_id=None):
        self.eval()
        B, T = ids.shape
        caches = None
        logits, _, caches = self.forward(ids, kv_caches=caches, offset=0)
        gen = ids
        off = T
        for _ in range(max_new):
            nl = logits[:, -1, :]
            if rep_pen != 1.0:
                for b in range(B):
                    for tid in set(gen[b].tolist()):
                        if nl[b, tid] > 0: nl[b, tid] /= rep_pen
                        else:              nl[b, tid] *= rep_pen
            if temp > 0:
                nl = nl / temp
                if top_k > 0:
                    tv, _ = torch.topk(nl, min(top_k, nl.size(-1)))
                    nl[nl < tv[:, -1:]] = float("-inf")
                if top_p < 1.0:
                    sl, si = torch.sort(nl, descending=True)
                    cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                    mask = (cp - F.softmax(sl, dim=-1)) >= top_p
                    sl[mask] = float("-inf")
                    nl = sl.scatter(1, si, sl)
                probs = F.softmax(nl, dim=-1)
                nxt = torch.multinomial(probs, 1)
            else:
                nxt = nl.argmax(dim=-1, keepdim=True)
            gen = torch.cat([gen, nxt], dim=1)
            if stop_id is not None and (nxt == stop_id).all(): break
            if gen.size(1) >= self.cfg.max_seq_len: break
            logits, _, caches = self.forward(nxt, kv_caches=caches, offset=off)
            off += 1
        return gen

model = LegalLM(CFG).to(DEVICE)
n_params = model.count_params()
print(f"Model: {n_params:,} parameters ({n_params/1e6:.1f}M)")
assert 50_000_000 <= n_params <= 300_000_000, f"Outside 50M–300M range: {n_params:,}"

# ── Smoke test ──
_x = torch.randint(0, CFG.vocab_size, (2, 32), device=DEVICE)
_l, _loss, _ = model(_x, targets=_x)
print(f"Smoke test: logits={_l.shape}, loss={_loss.item():.4f}")
_g = model.generate(_x[:1, :8], max_new=16, temp=1.0, top_k=10)
print(f"Generation smoke test: {_g.shape}")
del _x, _l, _loss, _g
torch.cuda.empty_cache()
print("✓ Model OK")

Model: 150,287,616 parameters (150.3M)
Smoke test: logits=torch.Size([2, 32, 40000]), loss=10.5437
Generation smoke test: torch.Size([1, 24])
✓ Model OK


## 7 — Training loop

In [25]:
def get_lr(step):
    if step < CFG.warmup_steps:
        return CFG.lr * (step + 1) / CFG.warmup_steps
    if step >= CFG.max_steps:
        return CFG.min_lr
    prog = (step - CFG.warmup_steps) / max(1, CFG.max_steps - CFG.warmup_steps)
    cos_val = 0.5 * (1.0 + math.cos(math.pi * prog))
    return CFG.min_lr + (CFG.lr - CFG.min_lr) * cos_val

@torch.no_grad()
def evaluate(model, dl, max_batches=50):
    model.eval()
    tot = n = 0
    for i, (x, y) in enumerate(dl):
        if i >= max_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            _, loss, _ = model(x, targets=y)
        tot += loss.item(); n += 1
    model.train()
    return tot / max(n, 1)

def save_ckpt(model, opt, scaler, step, val_loss, path):
    tmp_path = str(path) + ".tmp"
    torch.save({"step": step, "val_loss": val_loss,
                 "model": model.state_dict(),
                 "opt": opt.state_dict(),
                 "scaler": scaler.state_dict()}, tmp_path)
    os.replace(tmp_path, path)
    print(f"  💾 Saved {path}")

# ── Build optimizer ──
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if p.ndim < 2 or "norm" in name or "bias" in name:
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW([
    {"params": decay,    "weight_decay": CFG.weight_decay},
    {"params": no_decay, "weight_decay": 0.0},
], lr=CFG.lr, betas=CFG.betas)

scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)

# ── Resume? ──
start_step = 0
best_val = float("inf")
resume = CKPT_DIR / "best.pt"
if not resume.exists():
    step_ckpts = sorted(CKPT_DIR.glob("step_*.pt"),
                        key=lambda p: int(p.stem.split("_")[1]), reverse=True)
    if step_ckpts:
        resume = step_ckpts[0]
        print(f"best.pt not found, falling back to {resume.name}")

if resume.exists():
    try:
        ckpt = torch.load(resume, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["opt"])
        scaler.load_state_dict(ckpt["scaler"])
        start_step = ckpt["step"]
        best_val = ckpt.get("val_loss", float("inf"))
        print(f"Resumed from step {start_step}, best_val={best_val:.4f}")
    except RuntimeError as e:
        print(f"⚠ Checkpoint {resume} is corrupted: {e}")
        print("  Starting training from scratch.")
        model.apply(model._init)

print(f"\nTraining from step {start_step} to {CFG.max_steps}...")

/tmp/ipykernel_59/101664381.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)


best.pt not found, falling back to step_19000.pt
Resumed from step 19000, best_val=2.3696

Training from step 19000 to 20000...


In [26]:
model.train()
train_iter = iter(train_dl)
running = 0.0
t0 = time.time()
no_improve = 0          # early-stopping patience counter

for step in range(start_step, CFG.max_steps):
    lr = get_lr(step)
    for pg in optimizer.param_groups: pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    acc_loss = 0.0

    for _ in range(CFG.grad_accum):
        try: x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl)
            x, y = next(train_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            _, loss, _ = model(x, targets=y)
            loss = loss / CFG.grad_accum
        scaler.scale(loss).backward()
        acc_loss += loss.item()

    scaler.unscale_(optimizer)
    gn = nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
    scaler.step(optimizer)
    scaler.update()
    running += acc_loss

    if (step + 1) % CFG.log_interval == 0:
        avg = running / CFG.log_interval
        ppl = math.exp(min(avg, 20))
        el = time.time() - t0
        tps = CFG.log_interval * CFG.eff_batch * CFG.max_seq_len / el
        print(f"  step {step+1:6d}/{CFG.max_steps} | loss {avg:.4f} | "
              f"ppl {ppl:.1f} | lr {lr:.2e} | gn {gn:.2f} | "
              f"tok/s {tps:.0f} | {el:.0f}s")
        running = 0.0; t0 = time.time()

    if (step + 1) % CFG.eval_interval == 0:
        vl = evaluate(model, val_dl)
        vp = math.exp(min(vl, 20))
        print(f"  >>> val loss={vl:.4f}  ppl={vp:.1f}")
        if vl < best_val:
            best_val = vl
            no_improve = 0
            save_ckpt(model, optimizer, scaler, step+1, vl, CKPT_DIR/"best.pt")
        else:
            no_improve += 1
            print(f"  ⚠ No improvement for {no_improve}/{CFG.patience} evals")
            if no_improve >= CFG.patience:
                print(f"\n🛑 Early stopping at step {step+1} (patience={CFG.patience} exhausted)")
                save_ckpt(model, optimizer, scaler, step+1, best_val,
                          CKPT_DIR / f"early_stop_{step+1}.pt")
                break
        model.train()

    if (step + 1) % CFG.save_interval == 0:
        save_ckpt(model, optimizer, scaler, step+1, best_val,
                  CKPT_DIR / f"step_{step+1}.pt")
else:
    # Ran to max_steps without early stop
    save_ckpt(model, optimizer, scaler, CFG.max_steps, best_val, CKPT_DIR/"final.pt")

print(f"\n✓ Training complete. Best val loss: {best_val:.4f}")

/tmp/ipykernel_59/3190365867.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.use_amp):


  step  19050/20000 | loss 2.6823 | ppl 14.6 | lr 3.16e-05 | gn 0.83 | tok/s 6467 | 127s
  step  19100/20000 | loss 2.7361 | ppl 15.4 | lr 3.14e-05 | gn 0.90 | tok/s 6393 | 128s
  step  19150/20000 | loss 2.6889 | ppl 14.7 | lr 3.13e-05 | gn 0.85 | tok/s 6391 | 128s
  step  19200/20000 | loss 2.6744 | ppl 14.5 | lr 3.11e-05 | gn 0.87 | tok/s 6376 | 128s
  step  19250/20000 | loss 2.6536 | ppl 14.2 | lr 3.10e-05 | gn 0.82 | tok/s 6371 | 129s
  step  19300/20000 | loss 2.6679 | ppl 14.4 | lr 3.09e-05 | gn 0.82 | tok/s 6372 | 129s
  step  19350/20000 | loss 2.6773 | ppl 14.5 | lr 3.07e-05 | gn 0.89 | tok/s 6376 | 128s
  step  19400/20000 | loss 2.6538 | ppl 14.2 | lr 3.06e-05 | gn 0.90 | tok/s 6368 | 129s
  step  19450/20000 | loss 2.6675 | ppl 14.4 | lr 3.05e-05 | gn 0.83 | tok/s 6363 | 129s
  step  19500/20000 | loss 2.6911 | ppl 14.7 | lr 3.04e-05 | gn 0.87 | tok/s 6371 | 129s


/tmp/ipykernel_59/101664381.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.use_amp):


  >>> val loss=2.3670  ppl=10.7
  💾 Saved /kaggle/working/checkpoints/best.pt
  step  19550/20000 | loss 2.6692 | ppl 14.4 | lr 3.04e-05 | gn 0.83 | tok/s 5780 | 142s
  step  19600/20000 | loss 2.6888 | ppl 14.7 | lr 3.03e-05 | gn 0.88 | tok/s 6398 | 128s
  step  19650/20000 | loss 2.6861 | ppl 14.7 | lr 3.02e-05 | gn 0.84 | tok/s 6386 | 128s
  step  19700/20000 | loss 2.6608 | ppl 14.3 | lr 3.02e-05 | gn 0.81 | tok/s 6370 | 129s
  step  19750/20000 | loss 2.6623 | ppl 14.3 | lr 3.01e-05 | gn 0.85 | tok/s 6364 | 129s
  step  19800/20000 | loss 2.6156 | ppl 13.7 | lr 3.01e-05 | gn 0.82 | tok/s 6394 | 128s
  step  19850/20000 | loss 2.6640 | ppl 14.4 | lr 3.00e-05 | gn 0.85 | tok/s 6352 | 129s
  step  19900/20000 | loss 2.6981 | ppl 14.9 | lr 3.00e-05 | gn 0.88 | tok/s 6353 | 129s
  step  19950/20000 | loss 2.6789 | ppl 14.6 | lr 3.00e-05 | gn 0.86 | tok/s 6363 | 129s
  step  20000/20000 | loss 2.6870 | ppl 14.7 | lr 3.00e-05 | gn 0.84 | tok/s 6364 | 129s
  >>> val loss=2.3636  ppl=10.6


## 8 — Load best checkpoint

In [27]:
best_path = CKPT_DIR / "best.pt"
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded best checkpoint (step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f})")
else:
    print("No best checkpoint found — using final weights")

Loaded best checkpoint (step 20000, val_loss=2.3636)


## 9 — Generate predictions for Codabench

In [30]:
def generate_continuation(prompt, max_words=128):
    ids = tokenizer.encode(prompt)
    max_prompt = CFG.max_seq_len - 200
    if len(ids) > max_prompt: ids = ids[-max_prompt:]
    inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out = model.generate(inp, max_new=max_words * 3,
                         temp=CFG.gen_temperature, top_k=CFG.gen_top_k,
                         top_p=CFG.gen_top_p, rep_pen=CFG.gen_rep_penalty)
    new_ids = out[0, len(ids):].tolist()
    text = tokenizer.decode(new_ids)
    words = text.split()[:max_words]
    return " ".join(words)

# Load test
with open(test_path, "r") as f:
    test_data = json.load(f)

predictions = {}
total = len(test_data)
print(f"Generating continuations for {total} test prompts...")

model.eval()
for i, (tid, item) in enumerate(tqdm(test_data.items(), total=total, desc="Generating")):
    prompt = item["text"] if isinstance(item, dict) else item
    cont = generate_continuation(prompt, max_words=CFG.gen_max_words)
    predictions[tid] = {"text": cont}

# ── Sanity checks on predictions ──
assert len(predictions) == total, f"Missing predictions: {len(predictions)} vs {total}"
for tid, p in predictions.items():
    assert "text" in p, f"Missing 'text' for {tid}"
    assert len(p["text"].strip()) > 0, f"Empty text for {tid}"
word_counts = [len(p["text"].split()) for p in predictions.values()]
print(f"\n✓ All {total} predictions generated")
print(f"  Words: min={min(word_counts)}, max={max(word_counts)}, "
      f"mean={sum(word_counts)/len(word_counts):.1f}")

# ── Save ──
pred_path = OUT / "prediction.json"
with open(pred_path, "w") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

zip_path = OUT / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")

print(f"  {pred_path}")
print(f"  {zip_path}")

Generating continuations for 1000 test prompts...


Generating:   0%|          | 0/1000 [00:00<?, ?it/s]


✓ All 1000 predictions generated
  Words: min=53, max=128, mean=127.8
  /kaggle/working/prediction.json
  /kaggle/working/submission.zip


## 10 — Quick generation demo

In [31]:
demo_prompts = [
    "The Supreme Court held that the appellant was entitled to",
    "Section 302 of the Indian Penal Code provides that",
    "The learned counsel for the respondent submitted that the lower court",
]

model.eval()
for prompt in demo_prompts:
    cont = generate_continuation(prompt, max_words=40)
    print(f"PROMPT:  {prompt}")
    print(f"CONT:    {cont}")
    print()

PROMPT:  The Supreme Court held that the appellant was entitled to
CONT:    be treated as a workman under Section 25F of the Act. The appellant being aggrieved by the said decision of the Supreme Court preferred an appeal before the High Court. The High Court, however, rejected the same on the ground

PROMPT:  Section 302 of the Indian Penal Code provides that
CONT:    a person shall be deemed to be innocent until he has been so punished and unless he is proved to have committed an offence punishable with death or imprisonment for life, he shall be liable to be punished accordingly: Provided

PROMPT:  The learned counsel for the respondent submitted that the lower court
CONT:    had dismissed the application under Order 8 Rule 9 of the Code of Civil Procedure and the order passed by the trial Court was erroneous in law. The learned counsel submitted that the order of the High Court was perverse



## 11 — Output summary

| File | Description |
|---|---|
| `checkpoints/best.pt` | Best model checkpoint (lowest val loss) |
| `checkpoints/final.pt` | Final model checkpoint |
| `prediction.json` | Codabench predictions |
| `submission.zip` | Ready-to-submit archive |

Upload `submission.zip` to Codabench for evaluation.

In [32]:
import shutil

# ── Keep only what we need ──
best_ckpt = CKPT_DIR / "best.pt"
final_ckpt = CKPT_DIR / "final.pt"

# Pick best available checkpoint
keep_ckpt = best_ckpt if best_ckpt.exists() else final_ckpt
if not keep_ckpt.exists():
    step_ckpts = sorted(CKPT_DIR.glob("step_*.pt"),
                        key=lambda p: int(p.stem.split("_")[1]), reverse=True)
    if step_ckpts:
        keep_ckpt = step_ckpts[0]

assert keep_ckpt.exists(), "No checkpoint found!"
print(f"Keeping: {keep_ckpt.name} ({keep_ckpt.stat().st_size/1e6:.1f} MB)")

# Move checkpoint and prediction.json to working root
shutil.copy2(keep_ckpt, OUT / "best_model.pt")
assert pred_path.exists(), "prediction.json not found!"

# Remove everything else from /kaggle/working except what we need
keep = {"best_model.pt", "prediction.json", "submission.zip"}
for item in OUT.iterdir():
    if item.name in keep:
        continue
    print(f"  Removing: {item.name}")
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()

# ── Final zip with model + predictions ──
final_zip = OUT / "submission_with_model.zip"
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / "best_model.pt", "best_model.pt")
    zf.write(pred_path, "prediction.json")

print(f"\n✓ Cleaned /kaggle/working/")
print(f"  best_model.pt        {(OUT/'best_model.pt').stat().st_size/1e6:.1f} MB")
print(f"  prediction.json      {pred_path.stat().st_size/1e6:.1f} MB")
print(f"  submission.zip       {zip_path.stat().st_size/1e6:.1f} MB  (predictions only, for Codabench)")
print(f"  submission_with_model.zip  {final_zip.stat().st_size/1e6:.1f} MB  (model + predictions)")

Keeping: best.pt (1803.7 MB)
  Removing: .ipynb_checkpoints
  Removing: checkpoints
  Removing: bpe_40k.json
  Removing: .virtual_documents
  Removing: a1_tokenizer.py
  Removing: step_19000.zip

✓ Cleaned /kaggle/working/
  best_model.pt        1803.7 MB
  prediction.json      0.8 MB
  submission.zip       0.2 MB  (predictions only, for Codabench)
  submission_with_model.zip  1651.6 MB  (model + predictions)


In [33]:
!pip -q install rouge-score bert-score
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── Collect 20 long docs from Acts, split into 512-word prompt + 128-word reference ──
eval_prompts, eval_references = [], []
for rec in iter_records(LOCAL["Acts"]):
    raw = extract_text(rec)
    if raw is None: continue
    text = clean_text(raw)
    words = text.split()
    if len(words) < 700: continue
    eval_prompts.append(" ".join(words[:512]))
    eval_references.append(" ".join(words[512:512+128]))
    if len(eval_prompts) >= 20: break

print(f"Collected {len(eval_prompts)} eval samples")

# ── Generate continuations ──
eval_generated = []
model.eval()
for prompt in tqdm(eval_prompts, desc="Local eval"):
    gen = generate_continuation(prompt, max_words=128)
    eval_generated.append(gen)

# ── ROUGE-L ──
rscorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_scores = [rscorer.score(ref, gen)["rougeL"].fmeasure
                for gen, ref in zip(eval_generated, eval_references)]

# ── BLEU-4 ──
smooth = SmoothingFunction().method1
bleu_scores = [sentence_bleu([ref.split()], gen.split(),
               weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth)
               for gen, ref in zip(eval_generated, eval_references)]

# ── BERTScore ──
_, _, F1 = bert_score_fn(eval_generated, eval_references, lang="en",
                          verbose=False, batch_size=16)

print(f"\n{'='*50}")
print(f"  LOCAL EVAL RESULTS")
print(f"{'='*50}")
print(f"  BERTScore F1:  {F1.mean().item():.4f}")
print(f"  ROUGE-L F1:    {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"  BLEU-4:        {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"{'='*50}")
print("\nSample:")
print(f"  REF: {eval_references[0][:150]}...")
print(f"  GEN: {eval_generated[0][:150]}...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
Collected 20 eval samples


Local eval:   0%|          | 0/20 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



  LOCAL EVAL RESULTS
  BERTScore F1:  0.8450
  ROUGE-L F1:    0.2501
  BLEU-4:        0.0840

Sample:
  REF: anything contained in any judgement, decree or order of any court, tribunal or any other authority, Zonal regulations appended to the Outline Developm...
  GEN: anything contained in any other law for the time being in force, after the commencement of this Act, the following provisions shall have effect as if ...
